# Fixtures for the JavaScript port

> Python layouts that the JavaScript library must reproduce cell for cell.

The website uses [`js/itemiset.js`](https://github.com/adamcc/itemiset/blob/main/js/itemiset.js), a port of the [layout](layout.html) module. This notebook lays out a few hundred inputs with the Python reference and saves the results to `js/test/fixtures/layouts.json`. The JavaScript tests (`cd js && npm test`) lay out the same inputs and must get the same cells, roles, widths and notes.

When this notebook runs as a test it **checks** the saved file instead of rewriting it. If a change to the Python layout makes the file out of date, the test fails with a reminder to regenerate it:

```sh
ITEMISET_REGEN=1 nbdev-test --file-glob 04_fixtures.ipynb
```

and then the JavaScript port has to be updated until `npm test` passes again.

In [ ]:
import json, os, random
from pathlib import Path
from itemiset.layout import layout_sets, layout_record, check_layout
from itemiset.examples import load_example

## The cases

In [ ]:
def from_zones(spec, names=("Bait1", "Bait2", "Bait3")):
    "Sets from zone sizes, e.g. {'A': 3, 'AB': 2}. Items are named after their zone."
    code = dict(zip("ABC", names)); k = max(len(z) for z in spec) if spec else 1
    k = max(k, max("ABC".index(ch) + 1 for z in spec for ch in z))
    sets = {code[ch]: [] for ch in "ABC"[:k]}
    for z, n in spec.items():
        for j in range(n):
            for ch in z: sets[code[ch]].append(f"{z}_{j:02d}")
    return sets

GALLERY = {  # the prototype's edge-case gallery
    "all seven zones": {"A": 30, "B": 20, "C": 15, "AB": 8, "AC": 6, "BC": 10, "ABC": 5},
    "no triple overlap": {"A": 30, "B": 20, "C": 15, "AB": 8, "AC": 6, "BC": 10},
    "bait 3 disjoint": {"A": 30, "B": 20, "AB": 8, "C": 12},
    "all disjoint": {"A": 30, "B": 20, "C": 12},
    "bait 3 nested in 1 and 2": {"A": 30, "B": 20, "AB": 12, "ABC": 6},
    "single-member zones": {"A": 1, "B": 1, "ABC": 1},
    "bait 3 empty, 1 and 2 disjoint": {"A": 30, "B": 20, "C": 0},
    "filler needed": {"A": 3, "B": 3, "AB": 2, "AC": 3, "BC": 3},
}

cases = []
def add(name, sets, **opts):
    cases.append(dict(name=name, sets=[[k, list(v)] for k, v in sets.items()], options=opts))

for name, spec in GALLERY.items():
    sets = from_zones(spec)
    add(name, sets)
    add(name + ", Venn mode", sets, show_empty=True)
    if len(sets) == 3:
        for b in sets: add(f"{name}, {b} at the bottom", sets, bottom=b)

for ex in ("pbody", "decay"):
    sets = load_example(ex)["sets"]
    for t in (0.5, 1.0, 2.2, 3.0):
        add(f"{ex} example, aspect {t}", sets, target_aspect=t)
    add(f"{ex} example, Venn mode", sets, show_empty=True)
    add(f"{ex} example, two sets", dict(list(sets.items())[:2]))

add("one set", {"Only": ["b", "a", "c10", "c9"]})
add("two sets", {"L": ["x", "y", "z"], "R": ["z", "w"]})
add("natural sort", {"S": ["CNOT10", "cnot2", "CNOT1", "b02", "B2", "b1", "x", "X", "αβ", "ε", "14-3-3ε", "item007", "item7"]})
add("long names", {"S": ["ANKHD1-EIF4EBP3", "A", "B"], "T": ["A", "LONGER-NAME-STILL-HERE"]})
add("duplicates and an empty set", {"A": ["x", "x", "y"], "B": [], "C": ["y", "z", "z"]})
add("bottom set is the empty one", {"A": ["x", "y"], "B": ["y"], "C": []}, bottom="C")
add("explicit cell aspect", from_zones(GALLERY["all seven zones"]), cell_aspect=1.0)
add("small font", from_zones(GALLERY["all seven zones"]), fontsize=4)
add("max width 3", from_zones(GALLERY["no triple overlap"]), max_width=3)

rng = random.Random(20261007)
for t in range(160):
    k = rng.randint(1, 3)
    spec = {}
    for m in range(1, 2**k):
        z = "".join("ABC"[i] for i in range(k) if m >> i & 1)
        spec[z] = 0 if rng.random() < 0.35 else rng.randint(1, rng.choice([4, 12, 30]))
    sets = from_zones(spec, names=("Left", "Right", "Under"))
    if not any(sets.values()): continue
    opts = {}
    if rng.random() < 0.3: opts["show_empty"] = True
    if rng.random() < 0.5: opts["target_aspect"] = rng.choice([0.5, 0.8, 1.3, 2.2, 3.0])
    if k == 3 and rng.random() < 0.25: opts["bottom"] = rng.choice(list(sets))
    add(f"random {t}", sets, **opts)
len(cases)

205

Two inputs that must fail in both languages:

In [ ]:
errors = [dict(name="bottom set not in the input", sets=[["A", ["x"]], ["B", ["y"]], ["C", ["z"]]], options=dict(bottom="Q")),
          dict(name="four sets", sets=[["A", ["x"]], ["B", ["y"]], ["C", ["z"]], ["D", ["w"]]], options={})]
for e in errors:
    try: layout_sets(dict(e["sets"]), **e["options"]); raise AssertionError("expected an error")
    except ValueError: pass

## Laying them out

In [ ]:
for c in cases:
    lay = layout_sets(dict(c["sets"]), **c["options"])
    assert all(check_layout(lay, dict(c["sets"])).values()), c["name"]
    c["expected"] = layout_record(lay)

fixtures = dict(about="Written by nbs/04_fixtures.ipynb from the Python reference layout. Do not edit by hand.",
                version=1, cases=cases, errors=errors)
text = json.dumps(fixtures, ensure_ascii=False, separators=(",", ":")) + "\n"
f"{len(cases)} cases, {sum(len(c['expected']['cells']) for c in cases)} cells, {len(text)//1024} KB"

'205 cases, 6952 cells, 358 KB'

In [ ]:
path = Path("../js/test/fixtures/layouts.json")
if os.environ.get("ITEMISET_REGEN") or not path.exists():
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    print(f"Wrote {path}")
else:
    saved = path.read_text(encoding="utf-8")
    if saved != text:
        raise AssertionError("js/test/fixtures/layouts.json is out of date with the Python layout. "
                             "Regenerate it with `ITEMISET_REGEN=1 nbdev-test --file-glob 04_fixtures.ipynb`, "
                             "then update js/itemiset.js until `npm test` passes.")
    print("Fixtures are up to date.")

Fixtures are up to date.
